<h2 style="color: #0f2027; background: linear-gradient(90deg, #43cea2 0%, #185a9d 100%); padding: 12px 0; border-radius: 8px; text-align:center; font-size: 2rem; letter-spacing: 1px;">
   <span style="color: #fff;">Introduction to Ray</span> 
</h2>

## What is Ray ?

- **Open-source project** under PyTorch Foundation
- **Open-source distributed scheduler** for stateless tasks & stateful actors  
- **Key features:** task graphs, resource-aware, fast data transfer, GPU/custom resources  
- **Infra:** in-memory object store, fault-tolerant design  
- **Ecosystem:** Data, Train, Tune, Serve, RLlib  
- **User-friendly Python APIs**


<div align="center"><img src="assets/img01.png" alt="Intro to Ray" width="70%"></div>

| Concept | What It Is | Why It Matters (The Problem It Solves) |
| :--- | :--- | :--- |
| **`@ray.remote`** | A decorator to mark Python code for parallel execution. | The magic switch to turn a normal function or class into a distributed building block. |
| **Task** | A remote, stateless function call. | **Problem:** My code is slow because it only uses one core. A Task lets you run a function on any available core. |
| **Actor** | A remote, stateful class instance. | **Problem:** My parallel tasks need to share and update a common state (like a counter or a model). |
| **`.remote()`** | The syntax used to execute a Task or an Actor method. | The command to "send this work to the Ray cluster" instead of running it here. |
| **`ObjectRef`** | A "future" or a "receipt" for a result being computed. | The placeholder you get back instantly after calling `.remote()`, allowing your code to continue without waiting. |
| **`ray.get()`** | The command to retrieve the actual result from an `ObjectRef`. | **Problem:** My parallel work has been sent out; now I need the final answers back. |
| **`ray.put()`** | A command to place a large object into shared memory. | **Problem:** Sending the same large dataset (e.g., a big model) to every task is slow and wasteful. |

## Ray `Task`

<div align="center"><img src="assets/img02.png" alt="Ray Task" width="70%"></div>

#### Example of a sequential process (`without Ray`)

```python

# sequential_process.py
import time
import numpy as np



def process_image(image: np.ndarray) -> np.ndarray:
    """Simulates a slow 1-second filter."""
    time.sleep(1)
    return 255 - image

images = [np.random.randint(0, 255, (10, 10, 3)) for _ in range(8)]

start_time = time.time()

# Sequential: 8 images × 1 sec/image = 8 seconds
results = [process_image(img) for img in images]

end_time = time.time()

print(f"Processed {len(results)} images in {end_time - start_time:.2f} seconds.")

```

Let's `run` it!

In [1]:
!python code/sequential_process.py

Processed 8 images in 8.00 seconds.


#### Example of the same process but using `Ray Task`

Let's `run` it!

In [2]:
!python code/parallel_process.py

2026-01-08 17:46:26,026	INFO worker.py:1833 -- Connecting to existing Ray cluster at address: 10.0.36.30:6379...
2026-01-08 17:46:26,037	INFO worker.py:2004 -- Connected to Ray cluster. View the dashboard at https://session-zhee2uzsi3lhk3sdl5dvqc8x4m.i.anyscaleuserdata.com 
2026-01-08 17:46:26,066	INFO packaging.py:380 -- Pushing file package 'gcs://_ray_pkg_887788302c9d6520b5305e3e9acae68cd914d032.zip' (9.98MiB) to Ray cluster...
2026-01-08 17:46:26,102	INFO packaging.py:393 -- Successfully pushed file package 'gcs://_ray_pkg_887788302c9d6520b5305e3e9acae68cd914d032.zip'.
Processed 8 images in 40.38 seconds.


## Ray `Actors`

```python

import ray

# 1. Initialize Ray
if not ray.is_initialized():
    ray.init()

# 2. Define the Actor (Stateful Worker)
@ray.remote
class Counter:
    def __init__(self):
        self.count = 0  # <--- This is the "State"

    def increment(self):
        self.count += 1
        return self.count

    def get_count(self):
        return self.count

# Step 1:Create four Counter actors and increment each Counter once and get the results. These tasks all happen in parallel.
counters = [Counter.remote() for _ in range(4)]
results1 = ray.get([c.increment.remote() for c in counters])
print("Initial counts: ", results1)

# Step 2: Increment the first Counter five times. These tasks are executed sequentially and share state.
results2 = ray.get([counters[0].increment.remote() for _ in range(5)])
print("Incremented first counter five times: ", results2)
_ = ray.get([c.get_count.remote() for c in counters])
print("Get the counts: ", _)

# Step 3: Increment each Counter once and get the results. These tasks all happen in parallel.
results3 = ray.get([c.increment.remote() for c in counters])
print("Look at the final counts after incrementing each counter once: ", results3)

```

<div align="center"><img src="assets/ray_actors.jpg" alt="actors" width="80%"></div>


Let's `run` it!

In [4]:
!python code/ray_actor.py

2026-01-08 17:47:07,892	INFO worker.py:1833 -- Connecting to existing Ray cluster at address: 10.0.36.30:6379...
2026-01-08 17:47:07,902	INFO worker.py:2004 -- Connected to Ray cluster. View the dashboard at https://session-zhee2uzsi3lhk3sdl5dvqc8x4m.i.anyscaleuserdata.com 
2026-01-08 17:47:07,931	INFO packaging.py:380 -- Pushing file package 'gcs://_ray_pkg_887788302c9d6520b5305e3e9acae68cd914d032.zip' (9.98MiB) to Ray cluster...
2026-01-08 17:47:07,967	INFO packaging.py:393 -- Successfully pushed file package 'gcs://_ray_pkg_887788302c9d6520b5305e3e9acae68cd914d032.zip'.
Initial counts:  [1, 1, 1, 1]
Incremented first counter five times:  [2, 3, 4, 5, 6]
Get the counts:  [6, 1, 1, 1]
Look at the final counts after incrementing each counter once:  [7, 2, 2, 2]


In [5]:
# Same class, without Ray Actors 
!python code/counter.py

Initial counts:  [1, 1, 1, 1]
Incremented first counter five times:  [2, 3, 4, 5, 6]
Get the counts:  [6, 1, 1, 1]
Look at the final counts after incrementing each counter once:  [7, 2, 2, 2]


## 

We can see that the object has been garbage collected.

In [6]:
!ray list objects

/home/ray/anaconda3/lib/python3.12/site-packages/ray/util/state/api.py:413: UserWarning: Callsite is not being recorded. To record callsite information for each ObjectRef created, set env variable RAY_record_ref_creation_sites=1 during `ray start` and `ray.init`.
  warnings.warn(warning_to_print)
No resource in the cluster


## 